# 🧠 EX60: การทำนายผลแบบสตรีมมิ่ง (Streaming Inference)

`stream=False` เก็บทุกเฟรมในลิสต์ → RAM แบบ O(N) → OOM กับวิดีโอยาว
`stream=True` คืน Python **generator** → RAM แบบ O(1) คงที่

| | `stream=False` | `stream=True` |
|--|--|--|
| ประเภทคืนค่า | `list[Results]` | `Generator[Results]` |
| RAM | O(N frames) | O(1) |
| เข้าถึงด้วย index | ✅ `[5]` | ❌ |
| เหมาะกับ | คลิปสั้น | กล้อง CCTV 24/7 |

**Frame-drop:** ใช้ `vid_stride=N` ประมวลผลทุก N เฟรม เพื่อ real-time throughput

## 🔗 ลิงก์
- [[EX59_Multi_Task_Modes_TH]] | [[EX61_YOLO_Dataset_Format_TH]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os, types
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from solution import stream_inference_generator
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

video_path = "stream_demo.mp4"
out = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*"mp4v"), 15.0, (320,240))
for i in range(30):
    frame = np.zeros((240,320,3), dtype=np.uint8)
    cv2.circle(frame, (min(40+i*8,300),120), 25, (255,255,255), -1)
    cv2.putText(frame, f"F{i:02d}", (5,15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180,180,180), 1)
    out.write(frame)
out.release()

print("\n--- เริ่มการตรวจสอบ ---")
gen = stream_inference_generator("yolo11n.pt", video_path)
print(f"ประเภท: {type(gen).__name__} | is_generator={isinstance(gen, types.GeneratorType)}")

counts = []
for idx, dets in enumerate(gen):
    counts.append(len(dets))
    if idx < 5 or dets:
        print(f"  เฟรม {idx:02d}: {len(dets)} วัตถุ"
              + (f" | conf={dets[0]['confidence']:.3f}" if dets else ""))

print(f"\nรวม {len(counts)} เฟรม | detections: {sum(counts)}")
print("RAM: O(1) — generator ประมวลผลทีละเฟรม ✅")
print("--- สิ้นสุดการตรวจสอบ ---")

plt.figure(figsize=(10,4))
plt.bar(range(len(counts)), counts, color="#9b59b6", alpha=0.8)
plt.xlabel("เฟรม"); plt.ylabel("จำนวน Detections")
plt.title("ความหนาแน่นการตรวจจับในวิดีโอ")
plt.tight_layout(); plt.show()

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
if os.path.exists(video_path): os.remove(video_path)
